# Convert ModelWeights_RF to ONNX

Converts `ModelWeights_RF.joblib` to ONNX format for browser-based inference via `onnxruntime-web`.  
Also extracts the OrdinalEncoder category mappings and SimpleImputer medians, saving them as `model_meta.json` for use by the JavaScript frontend.

**Run this notebook once after training.  
Outputs (saved to OUTPUT_DIR):**
- `ModelWeights_RF.onnx` — model in ONNX format, loadable by the browser
- `model_meta.json` — encoder categories + imputer medians for JavaScript preprocessing

**Required packages:**
```
pip install skl2onnx onnx onnxruntime
```

## 0. Install Dependencies

In [ ]:
import sys
!{sys.executable} -m pip install onnxmltools --quiet
!{sys.executable} -m pip install skl2onnx onnx onnxruntime

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1.1
[notice] To update, run: C:\Users\flori\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## 1. Imports & Configuration

In [ ]:
import json
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

OUTPUT_DIR = Path(r'C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output')

print('Output directory:', OUTPUT_DIR)
print('joblib file exists:', (OUTPUT_DIR / 'ModelWeights_Quantile.joblib').exists())

Output directory: C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output
joblib file exists: True


## 2. Load Pipeline

In [ ]:
pipeline = joblib.load(OUTPUT_DIR / 'ModelWeights_Quantile.joblib')

preprocessor = pipeline.named_steps['prep']
rf            = pipeline.named_steps['rf']

print('Pipeline loaded.')
print(f'  Trees        : {rf.n_estimators}')
print(f'  Max depth    : {rf.max_depth}')
print(f'  Min leaf     : {rf.min_samples_leaf}')
print(f'  Max features : {rf.max_features}')
print(f'  Transformers : {[name for name, _, _ in preprocessor.transformers]}')

Pipeline loaded.
  Trees        : 400
  Max depth    : 20
  Min leaf     : 5
  Max features : sqrt
  Transformers : ['cat', 'num']


## 3. Auto-Detect Feature Names

Reads `cat_cols` and `num_cols` directly from the pipeline's `ColumnTransformer`
so the rest of the notebook works correctly regardless of which model version is loaded.

In [5]:
ct = pipeline.named_steps['prep']
cat_cols = []
num_cols = []
for name, _, cols in ct.transformers:
    col_list = cols if isinstance(cols, list) else [cols]
    if name == 'cat':
        cat_cols.extend(col_list)
    elif name == 'num':
        num_cols.extend(col_list)

print('Categorical features:', cat_cols)
print('Numeric features    :', num_cols)

Categorical features: ['permittypedesc', 'permitclass', 'zone_family']
Numeric features    : ['log_estprojectcost', 'log_housingunitsadded', 'latitude', 'longitude']


## 4. Extract OrdinalEncoder Categories

The OrdinalEncoder learned its category-to-integer mapping from training data.  
We extract that mapping here so the JavaScript frontend can replicate it exactly.

In [6]:
cat_transformer = preprocessor.named_transformers_['cat']
ordinal_enc     = cat_transformer.named_steps['encode']

# categories_ is a list of arrays, one per categorical feature
# Order matches cat_cols exactly
all_categories = {
    col: ordinal_enc.categories_[i].tolist()
    for i, col in enumerate(cat_cols)
}

for col, cats in all_categories.items():
    print(f'{col} categories ({len(cats)}):')
    for i, cat in enumerate(cats):
        print(f'  {i:>2}  {cat}')
    print()

permittypedesc categories (6):
   0  Addition/Alteration
   1  Change of Use Only - No Construction
   2  Deconstruction
   3  Demolition
   4  New
   5  Unknown

permitclass categories (5):
   0  Commercial
   1  Institutional
   2  Multifamily
   3  Single Family/Duplex
   4  Vacant Land

zone_family categories (21):
   0  C1
   1  C2
   2  DH
   3  DMC
   4  DOC
   5  HR
   6  IB
   7  IC
   8  IDM
   9  IG
  10  LR
  11  MIO
  12  MR
  13  NC
  14  NR
  15  Other
  16  PMM
  17  PSM
  18  SF
  19  SM
  20  Unknown



## 5. Extract SimpleImputer Medians

Null values in numeric features are replaced with the median computed during training.  
We extract those medians so the JavaScript can apply the same imputation.

In [7]:
num_transformer = preprocessor.named_transformers_['num']
imputer         = num_transformer.named_steps['impute']
medians         = imputer.statistics_.tolist()
num_features    = num_cols  # use auto-detected list from cell 3

print('Imputer medians:')
for feat, med in zip(num_features, medians):
    print(f'  {feat:<28}  {med:.6f}')

Imputer medians:
  log_estprojectcost            12.018230
  log_housingunitsadded         0.000000
  latitude                      47.640208
  longitude                     -122.334014


## 6. Save model_meta.json

This file is loaded by `index.html` at runtime so the JavaScript preprocessing
mirrors the Python pipeline exactly.

In [8]:
meta = {
    'cat_features': cat_cols,
    'num_features': num_features,
    'categories': all_categories,
    'imputer_medians': dict(zip(num_features, medians)),
    'unknown_value': -1,
    'note': 'OrdinalEncoder: category index = position in each categories list. Unknown values get -1.'
}

meta_path = OUTPUT_DIR / 'model_meta.json'
with open(meta_path, 'w') as f:
    json.dump(meta, f, indent=2)

print(f'Saved: {meta_path}')
print()
print(json.dumps(meta, indent=2))

Saved: C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\model_meta.json

{
  "cat_features": [
    "permittypedesc",
    "permitclass",
    "zone_family"
  ],
  "num_features": [
    "log_estprojectcost",
    "log_housingunitsadded",
    "latitude",
    "longitude"
  ],
  "categories": {
    "permittypedesc": [
      "Addition/Alteration",
      "Change of Use Only - No Construction",
      "Deconstruction",
      "Demolition",
      "New",
      "Unknown"
    ],
    "permitclass": [
      "Commercial",
      "Institutional",
      "Multifamily",
      "Single Family/Duplex",
      "Vacant Land"
    ],
    "zone_family": [
      "C1",
      "C2",
      "DH",
      "DMC",
      "DOC",
      "HR",
      "IB",
      "IC",
      "IDM",
      "IG",
      "LR",
      "MIO",
      "MR",
      "NC",
      "NR",
      "Other",
      "PMM",
      "PSM",
      "SF",
      "SM",
      "Unknown"
    ]
  },
  "imputer_medians": {
    "log_estprojectcost": 12.01822987484

## 7. Convert Pipeline to ONNX

Uses `skl2onnx` to translate the full scikit-learn pipeline (preprocessor + random forest)
into the ONNX (Open Neural Network Exchange) graph format.  
The resulting `.onnx` file can be loaded and run entirely in the browser via `onnxruntime-web`.

In [ ]:
from skl2onnx import convert_sklearn
from onnxmltools.convert import convert_xgboost
from skl2onnx.common.data_types import FloatTensorType, StringTensorType

# Build initial_types dynamically from auto-detected feature lists
initial_types = (
    [(col, StringTensorType([None, 1])) for col in cat_cols] +
    [(col, FloatTensorType([None, 1]))  for col in num_cols]
)

print('initial_types:')
for name, typ in initial_types:
    print(f'  {name:<30} {type(typ).__name__}')

print('\nConverting pipeline to ONNX (this may take 1-2 minutes for a 400-tree forest)...')

onnx_model = convert_sklearn(
    pipeline,
    initial_types=initial_types,
    target_opset=17
)

onnx_path = OUTPUT_DIR / 'ModelWeights_RF.onnx'
with open(onnx_path, 'wb') as f:
    f.write(onnx_model.SerializeToString())

size_mb = onnx_path.stat().st_size / (1024 * 1024)
print(f'\nSaved: {onnx_path}')
print(f'File size: {size_mb:.1f} MB')

initial_types:
  permittypedesc                 StringTensorType
  permitclass                    StringTensorType
  zone_family                    StringTensorType
  log_estprojectcost             FloatTensorType
  log_housingunitsadded          FloatTensorType
  latitude                       FloatTensorType
  longitude                      FloatTensorType

Converting pipeline to ONNX (this may take 1-2 minutes for a 400-tree forest)...

Saved: C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\ModelWeights_RF.onnx
File size: 26.4 MB


## 8. Verify ONNX Output

Runs a test inference locally using `onnxruntime` to confirm the ONNX model
produces the same output as the original joblib pipeline.

In [10]:
import onnxruntime as rt

sess = rt.InferenceSession(str(onnx_path))

print('ONNX session inputs:')
for inp in sess.get_inputs():
    print(f'  {inp.name:<30} shape={inp.shape}  type={inp.type}')

print('\nONNX session outputs:')
for out in sess.get_outputs():
    print(f'  {out.name:<30} shape={out.shape}  type={out.type}')

ONNX session inputs:
  permittypedesc                 shape=[None, 1]  type=tensor(string)
  permitclass                    shape=[None, 1]  type=tensor(string)
  zone_family                    shape=[None, 1]  type=tensor(string)
  log_estprojectcost             shape=[None, 1]  type=tensor(float)
  log_housingunitsadded          shape=[None, 1]  type=tensor(float)
  latitude                       shape=[None, 1]  type=tensor(float)
  longitude                      shape=[None, 1]  type=tensor(float)

ONNX session outputs:
  variable                       shape=[None, 1]  type=tensor(float)


In [11]:
# Test values covering all possible features
# Only the ones present in cat_cols / num_cols will be used
test_values = {
    'permittypedesc':        'Addition/Alteration',
    'permitclass':           'Single Family/Duplex',
    'zone_family':           'NR',
    'log_estprojectcost':    np.log1p(250_000),
    'log_housingunitsadded': np.log1p(1),
    'latitude':              47.60,
    'longitude':             -122.32,
}

# Build test row with only the features the pipeline expects
test_row = pd.DataFrame([{k: test_values[k] for k in cat_cols + num_cols}])
print('Test row:')
print(test_row)

# joblib prediction
joblib_pred_log  = pipeline.predict(test_row)[0]
joblib_pred_days = np.expm1(joblib_pred_log)

# ONNX prediction — build inputs dynamically from detected feature lists
onnx_inputs = {}
for col in cat_cols:
    onnx_inputs[col] = np.array([[test_values[col]]], dtype=object)
for col in num_cols:
    onnx_inputs[col] = np.array([[float(test_values[col])]], dtype=np.float32)

onnx_result    = sess.run(None, onnx_inputs)
onnx_pred_log  = float(onnx_result[0][0])
onnx_pred_days = np.expm1(onnx_pred_log)

print('\n=== Prediction Comparison ===')
print(f'  joblib pipeline  : {joblib_pred_days:.1f} days  (log={joblib_pred_log:.4f})')
print(f'  ONNX model       : {onnx_pred_days:.1f} days  (log={onnx_pred_log:.4f})')
print(f'  Difference       : {abs(joblib_pred_days - onnx_pred_days):.2f} days')

if abs(joblib_pred_log - onnx_pred_log) < 0.01:
    print('\n✓ ONNX output matches joblib pipeline. Ready for deployment.')
else:
    print('\n⚠ Outputs differ by more than 0.01 log units. Check skl2onnx version compatibility.')

Test row:
        permittypedesc           permitclass zone_family  log_estprojectcost  \
0  Addition/Alteration  Single Family/Duplex          NR            12.42922   

   log_housingunitsadded  latitude  longitude  
0               0.693147      47.6    -122.32  

=== Prediction Comparison ===
  joblib pipeline  : 118.7 days  (log=4.7852)
  ONNX model       : 118.7 days  (log=4.7852)
  Difference       : 0.00 days

✓ ONNX output matches joblib pipeline. Ready for deployment.


C:\Users\flori\AppData\Local\Temp\ipykernel_14840\1285380932.py:30: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  onnx_pred_log  = float(onnx_result[0][0])


## 9. Summary

In [12]:
print('Files ready for deployment:')
for fname in ['ModelWeights_RF.onnx', 'model_meta.json', 'permit_lookup.json']:
    p = OUTPUT_DIR / fname
    if p.exists():
        size = p.stat().st_size / (1024 * 1024)
        print(f'  ✓ {fname:<30} {size:.2f} MB')
    else:
        print(f'  ✗ {fname:<30} NOT FOUND')

print()
print('GitHub repository root should contain:')
for f in ['index.html', 'styles.css', 'ModelWeights_RF.onnx', 'model_meta.json', 'permit_lookup.json']:
    print(f'  {f}')


Files ready for deployment:
  ✓ ModelWeights_RF.onnx           26.43 MB
  ✓ model_meta.json                0.00 MB
  ✗ permit_lookup.json             NOT FOUND

GitHub repository root should contain:
  index.html
  styles.css
  ModelWeights_RF.onnx
  model_meta.json
  permit_lookup.json


## 10. Generate permit_lookup.json

Exports a spatial grid index of all permits with recorded review times for use by the demo webpage.  
When a user enters an address, the page geocodes it and checks this file for nearby historical permits,  
then displays the model prediction alongside the actual recorded review time for comparison.

Field names are abbreviated to minimize file size:
- `p` = permitnum, `a` = address, `la/lo` = lat/lon
- `d` = days review, `t` = permit type, `c` = permit class
- `z` = zone family, `co` = estimated cost, `y` = application year

In [13]:
import pandas as pd
import json

MASTER_CSV = OUTPUT_DIR.parent / 'data' / 'master_dataset.csv'

df_lookup = pd.read_csv(MASTER_CSV, usecols=[
    'permitnum', 'originaladdress1', 'latitude', 'longitude',
    'totaldaysplanreview', 'permittypedesc', 'permitclass',
    'zone_family', 'estprojectcost', 'app_year'
])

df_lookup = df_lookup.dropna(subset=['latitude', 'longitude', 'totaldaysplanreview', 'originaladdress1'])
df_lookup['latitude']            = df_lookup['latitude'].round(5)
df_lookup['longitude']           = df_lookup['longitude'].round(5)
df_lookup['totaldaysplanreview'] = df_lookup['totaldaysplanreview'].astype(int)
df_lookup['estprojectcost']  = df_lookup['estprojectcost'].fillna(0).astype(int)
df_lookup['permittypedesc']  = df_lookup['permittypedesc'].fillna('Unknown')
df_lookup['permitclass']     = df_lookup['permitclass'].fillna('Unknown')
df_lookup['zone_family']     = df_lookup['zone_family'].fillna('Unknown')
df_lookup['grid_lat']            = df_lookup['latitude'].round(3)
df_lookup['grid_lon']            = df_lookup['longitude'].round(3)

# Build spatial grid index — each key is a ~110m grid cell
# containing a list of all permits in that area
index = {}
for _, row in df_lookup.iterrows():
    key = f"{row.grid_lat:.3f},{row.grid_lon:.3f}"
    rec = {
        'p':  row.permitnum,
        'a':  row.originaladdress1,
        'la': row.latitude,
        'lo': row.longitude,
        'd':  row.totaldaysplanreview,
        't':  row.permittypedesc,
        'c':  row.permitclass,
        'z':  row.zone_family,
        'co': row.estprojectcost,
        'y':  int(row.app_year),
    }
    if key not in index:
        index[key] = []
    index[key].append(rec)

lookup_path = OUTPUT_DIR / 'permit_lookup.json'
with open(lookup_path, 'w') as f:
    json.dump(index, f, separators=(',', ':'))

size_kb = lookup_path.stat().st_size / 1024
print(f'Grid cells : {len(index):,}')
print(f'Permits    : {len(df_lookup):,}')
print(f'File size  : {size_kb:.0f} KB ({size_kb/1024:.2f} MB)')
print(f'Saved      : {lookup_path}')

Grid cells : 8,074
Permits    : 14,182
File size  : 2305 KB (2.25 MB)
Saved      : C:\Users\flori\Documents\GitHub\CSB425-City-of-Seattle-Permit-Predictor\output\permit_lookup.json
